# Validate Silver configuration mappings

## Purpose

Validate the five versioned Silver configuration tables before any transformation consumes them. Structural problems are `stop` checks. Unmapped Bronze values, pending decisions, and an empty manual-override table are review `flag` checks. The notebook records results without changing Bronze or the mapping rules.

This milestone uses a separate `04-validation.silver_config_dq_results` table. Extending the existing Bronze result schema could break its current writer because the Bronze notebook does not supply Silver lineage fields.

## Step 1 — Prepare validation results and check table presence

The results table is append-only validation evidence. Missing configuration tables fail before any rule query attempts to read them.

In [ ]:
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

CREATE TABLE IF NOT EXISTS `04-validation`.silver_config_dq_results (
    run_id STRING NOT NULL,
    layer STRING NOT NULL,
    table_name STRING NOT NULL,
    mapping_version STRING,
    data_quality_attribute STRING NOT NULL,
    data_quality_check STRING NOT NULL,
    failed_rows BIGINT NOT NULL,
    total_rows BIGINT NOT NULL,
    status STRING NOT NULL,
    action STRING NOT NULL,
    details STRING,
    run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Silver configuration validation history with mapping-version lineage';

In [ ]:
WITH required_tables AS (
    SELECT *
    FROM VALUES
        ('config_place_name_alias'),
        ('config_project_category_mapping'),
        ('config_project_status_mapping'),
        ('config_mgb_susceptibility_mapping'),
        ('config_manual_geographic_match')
    AS expected(table_name)
),
missing_tables AS (
    SELECT expected.table_name
    FROM required_tables AS expected
    LEFT JOIN information_schema.tables AS actual
        ON actual.table_catalog = 'buildabida-capstone'
        AND actual.table_schema = '02-silver'
        AND actual.table_name = expected.table_name
    WHERE actual.table_name IS NULL
)
SELECT ASSERT_TRUE(
    COUNT(*) = 0,
    CONCAT('Missing required Silver configuration tables: ', CONCAT_WS(', ', COLLECT_LIST(table_name)))
)
FROM missing_tables;

## Step 2 — Evaluate structural and governance rules

These checks cover consistency, accuracy limits, auditability, validity, uniqueness, and timeliness. Approved geographic targets must exist in Bronze PSGC. Mapping versions and approval states are always explicit.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW silver_config_structural_checks AS
WITH psgc_reference AS (
    SELECT DISTINCT TRIM(psgc_code) AS psgc_code
    FROM `01-bronze`.psgc
)
SELECT
    'uniqueness' AS data_quality_attribute,
    'config_place_name_alias' AS table_name,
    COALESCE(MAX(alias_version), 'NO_RULES_YET') AS mapping_version,
    'natural key is unique' AS data_quality_check,
    'stop' AS action,
    (
        SELECT COUNT(*)
        FROM (
            SELECT 1
            FROM `02-silver`.config_place_name_alias
            GROUP BY source_system, raw_place_name, raw_region_name,
                raw_province_name, raw_city_municipality_name, place_type, alias_version
            HAVING COUNT(*) > 1
        ) AS duplicates
    ) AS failed_rows,
    COUNT(*) AS total_rows,
    'One context-specific source alias per alias version' AS details
FROM `02-silver`.config_place_name_alias
UNION ALL
SELECT
    'uniqueness',
    'config_project_category_mapping',
    COALESCE(MAX(taxonomy_version), 'NO_RULES_YET'),
    'natural key is unique',
    'stop',
    (
        SELECT COUNT(*)
        FROM (
            SELECT 1
            FROM `02-silver`.config_project_category_mapping
            GROUP BY source_system, raw_category, source_infra_type, taxonomy_version
            HAVING COUNT(*) > 1
        ) AS duplicates
    ),
    COUNT(*),
    'One source category and infrastructure-type pair per taxonomy version'
FROM `02-silver`.config_project_category_mapping
UNION ALL
SELECT
    'consistency',
    'config_project_category_mapping',
    COALESCE(MAX(taxonomy_version), 'NO_RULES_YET'),
    'exactly one active approved DPWH taxonomy version is selected',
    'stop',
    ABS(COUNT(DISTINCT CASE WHEN source_system = 'dpwh_projects' AND approval_status = 'APPROVED' AND is_active THEN taxonomy_version END) - 1),
    1,
    'DPWH component-category rules must resolve through one approved active version'
FROM `02-silver`.config_project_category_mapping
UNION ALL
SELECT
    'uniqueness',
    'config_project_category_mapping',
    COALESCE(MAX(taxonomy_version), 'NO_RULES_YET'),
    'DPWH category rules cannot fan out one source classification',
    'stop',
    (
        SELECT COUNT(*)
        FROM (
            SELECT source_system, raw_category, source_infra_type
            FROM `02-silver`.config_project_category_mapping
            WHERE approval_status = 'APPROVED' AND is_active
            GROUP BY source_system, raw_category, source_infra_type
            HAVING COUNT(*) > 1
        ) AS duplicates
    ),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    'One active approved mapping may match each complete category key'
FROM `02-silver`.config_project_category_mapping
UNION ALL
SELECT
    'validity',
    'config_project_category_mapping',
    COALESCE(MAX(taxonomy_version), 'NO_RULES_YET'),
    'approved DPWH category targets and flood markers are complete',
    'stop',
    COUNT_IF(approval_status = 'APPROVED' AND is_active AND (NULLIF(TRIM(standardized_sector), '') IS NULL OR is_flood_related IS NULL)),
    COUNT_IF(approval_status = 'APPROVED' AND is_active),
    'Every active approved source classification needs a sector and flood meaning'
FROM `02-silver`.config_project_category_mapping
UNION ALL
SELECT
    'uniqueness',
    'config_project_status_mapping',
    COALESCE(MAX(status_mapping_version), 'NO_RULES_YET'),
    'natural key is unique',
    'stop',
    (
        SELECT COUNT(*)
        FROM (
            SELECT 1
            FROM `02-silver`.config_project_status_mapping
            GROUP BY source_system, source_status, status_mapping_version
            HAVING COUNT(*) > 1
        ) AS duplicates
    ),
    COUNT(*),
    'One source status per mapping version'
FROM `02-silver`.config_project_status_mapping
UNION ALL
SELECT
    'uniqueness',
    'config_mgb_susceptibility_mapping',
    COALESCE(MAX(mapping_version), 'NO_RULES_YET'),
    'natural key is unique',
    'stop',
    (
        SELECT COUNT(*)
        FROM (
            SELECT 1
            FROM `02-silver`.config_mgb_susceptibility_mapping
            GROUP BY source_system, raw_susceptibility_code, source_version, mapping_version
            HAVING COUNT(*) > 1
        ) AS duplicates
    ),
    COUNT(*),
    'One raw MGB code per source-contract and mapping version'
FROM `02-silver`.config_mgb_susceptibility_mapping
UNION ALL
SELECT
    'uniqueness',
    'config_manual_geographic_match',
    COALESCE(MAX(mapping_version), 'NO_RULES_YET'),
    'natural key is unique',
    'stop',
    (
        SELECT COUNT(*)
        FROM (
            SELECT 1
            FROM `02-silver`.config_manual_geographic_match
            GROUP BY source_system, record_type, source_record_id, mapping_version
            HAVING COUNT(*) > 1
        ) AS duplicates
    ),
    COUNT(*),
    'One source-record override per mapping version'
FROM `02-silver`.config_manual_geographic_match
UNION ALL
SELECT
    'validity',
    'all_config_tables',
    'mixed',
    'approval status uses a controlled value',
    'stop',
    SUM(invalid_rows),
    SUM(total_rows),
    'Allowed values are APPROVED, PENDING_REVIEW, and DEPRECATED'
FROM (
    SELECT COUNT_IF(approval_status NOT IN ('APPROVED', 'PENDING_REVIEW', 'DEPRECATED')) AS invalid_rows, COUNT(*) AS total_rows FROM `02-silver`.config_place_name_alias
    UNION ALL SELECT COUNT_IF(approval_status NOT IN ('APPROVED', 'PENDING_REVIEW', 'DEPRECATED')), COUNT(*) FROM `02-silver`.config_project_category_mapping
    UNION ALL SELECT COUNT_IF(approval_status NOT IN ('APPROVED', 'PENDING_REVIEW', 'DEPRECATED')), COUNT(*) FROM `02-silver`.config_project_status_mapping
    UNION ALL SELECT COUNT_IF(approval_status NOT IN ('APPROVED', 'PENDING_REVIEW', 'DEPRECATED')), COUNT(*) FROM `02-silver`.config_mgb_susceptibility_mapping
    UNION ALL SELECT COUNT_IF(approval_status NOT IN ('APPROVED', 'PENDING_REVIEW', 'DEPRECATED')), COUNT(*) FROM `02-silver`.config_manual_geographic_match
) AS approval_checks
UNION ALL
SELECT
    'timeliness',
    'all_config_tables',
    'mixed',
    'mapping version is present',
    'stop',
    SUM(invalid_rows),
    SUM(total_rows),
    'Every controlled rule carries its mapping or taxonomy version'
FROM (
    SELECT COUNT_IF(TRIM(alias_version) = '') AS invalid_rows, COUNT(*) AS total_rows FROM `02-silver`.config_place_name_alias
    UNION ALL SELECT COUNT_IF(TRIM(taxonomy_version) = ''), COUNT(*) FROM `02-silver`.config_project_category_mapping
    UNION ALL SELECT COUNT_IF(TRIM(status_mapping_version) = ''), COUNT(*) FROM `02-silver`.config_project_status_mapping
    UNION ALL SELECT COUNT_IF(TRIM(mapping_version) = '' OR TRIM(source_version) = ''), COUNT(*) FROM `02-silver`.config_mgb_susceptibility_mapping
    UNION ALL SELECT COUNT_IF(TRIM(mapping_version) = ''), COUNT(*) FROM `02-silver`.config_manual_geographic_match
) AS version_checks
UNION ALL
SELECT
    'auditability',
    'all_config_tables',
    'mixed',
    'approved rules retain decision evidence',
    'stop',
    SUM(invalid_rows),
    SUM(total_rows),
    'Approved rows require a mapping rule or override reason and a source reference'
FROM (
    SELECT COUNT_IF(approval_status = 'APPROVED' AND (TRIM(COALESCE(match_rule, '')) = '' OR TRIM(COALESCE(source_reference, '')) = '')) AS invalid_rows, COUNT(*) AS total_rows FROM `02-silver`.config_place_name_alias
    UNION ALL SELECT COUNT_IF(approval_status = 'APPROVED' AND (TRIM(COALESCE(mapping_rule, '')) = '' OR TRIM(COALESCE(source_reference, '')) = '')), COUNT(*) FROM `02-silver`.config_project_category_mapping
    UNION ALL SELECT COUNT_IF(approval_status = 'APPROVED' AND (TRIM(COALESCE(mapping_rule, '')) = '' OR TRIM(COALESCE(source_reference, '')) = '')), COUNT(*) FROM `02-silver`.config_project_status_mapping
    UNION ALL SELECT COUNT_IF(approval_status = 'APPROVED' AND (TRIM(COALESCE(mapping_rule, '')) = '' OR TRIM(COALESCE(source_reference, '')) = '')), COUNT(*) FROM `02-silver`.config_mgb_susceptibility_mapping
    UNION ALL SELECT COUNT_IF(approval_status = 'APPROVED' AND (TRIM(COALESCE(override_reason, '')) = '' OR TRIM(COALESCE(source_reference, '')) = '')), COUNT(*) FROM `02-silver`.config_manual_geographic_match
) AS audit_checks
UNION ALL
SELECT
    'validity',
    'config_project_category_mapping',
    COALESCE(MAX(taxonomy_version), 'NO_RULES_YET'),
    'at least one source category field is present',
    'stop',
    COUNT_IF(TRIM(raw_category) = '' AND TRIM(source_infra_type) = ''),
    COUNT(*),
    'An empty pair cannot identify a category rule'
FROM `02-silver`.config_project_category_mapping
UNION ALL
SELECT
    'accuracy',
    'config_project_status_mapping',
    COALESCE(MAX(status_mapping_version), 'NO_RULES_YET'),
    'unsupported delayed target is absent',
    'stop',
    COUNT_IF(LOWER(TRIM(COALESCE(standardized_status, ''))) = 'delayed'),
    COUNT(*),
    'Long-running and stalled are future derived rules, not status mappings'
FROM `02-silver`.config_project_status_mapping
UNION ALL
SELECT
    'validity',
    'config_mgb_susceptibility_mapping',
    COALESCE(MAX(mapping_version), 'NO_RULES_YET'),
    'approved MGB levels and ranks are controlled',
    'stop',
    COUNT_IF(
        approval_status = 'APPROVED' AND is_active AND (
            standardized_level NOT IN ('Low', 'Moderate', 'High', 'Very High')
            OR severity_rank NOT BETWEEN 1 AND 4
            OR (standardized_level = 'Low' AND severity_rank <> 1)
            OR (standardized_level = 'Moderate' AND severity_rank <> 2)
            OR (standardized_level = 'High' AND severity_rank <> 3)
            OR (standardized_level = 'Very High' AND severity_rank <> 4)
        )
    ),
    COUNT(*),
    'Approved levels use the documented four-level order'
FROM `02-silver`.config_mgb_susceptibility_mapping
UNION ALL
SELECT
    'completeness',
    'config_mgb_susceptibility_mapping',
    COALESCE(MAX(mapping_version), 'NO_RULES_YET'),
    'four documented active MGB mappings are present',
    'stop',
    ABS(COUNT_IF(approval_status = 'APPROVED' AND is_active) - 4),
    4,
    'Only LF, MF, HF, and VHF are approved in the initial release'
FROM `02-silver`.config_mgb_susceptibility_mapping
UNION ALL
SELECT
    'accuracy',
    'config_mgb_susceptibility_mapping',
    COALESCE(MAX(mapping_version), 'NO_RULES_YET'),
    'blank and no-rating values are not approved automatically',
    'stop',
    COUNT_IF(approval_status = 'APPROVED' AND UPPER(TRIM(raw_susceptibility_code)) IN ('', 'NO RATING')),
    COUNT(*),
    'Unknown source values require a separate human decision'
FROM `02-silver`.config_mgb_susceptibility_mapping
UNION ALL
SELECT
    'consistency',
    'config_place_name_alias',
    COALESCE(MAX(config.alias_version), 'NO_RULES_YET'),
    'approved PSGC target exists in Bronze reference',
    'stop',
    COUNT_IF(config.approval_status = 'APPROVED' AND config.is_active AND reference.psgc_code IS NULL),
    COUNT(*),
    'Approved aliases resolve to a PSGC code in the selected Bronze snapshot'
FROM `02-silver`.config_place_name_alias AS config
LEFT JOIN psgc_reference AS reference
    ON TRIM(config.canonical_psgc_code) = reference.psgc_code
UNION ALL
SELECT
    'consistency',
    'config_manual_geographic_match',
    COALESCE(MAX(config.mapping_version), 'NO_RULES_YET'),
    'approved PSGC target exists in Bronze reference',
    'stop',
    COUNT_IF(config.approval_status = 'APPROVED' AND config.is_active AND reference.psgc_code IS NULL),
    COUNT(*),
    'Approved overrides resolve to a PSGC code in the selected Bronze snapshot'
FROM `02-silver`.config_manual_geographic_match AS config
LEFT JOIN psgc_reference AS reference
    ON TRIM(config.canonical_psgc_code) = reference.psgc_code
UNION ALL
SELECT
    'auditability',
    'config_manual_geographic_match',
    COALESCE(MAX(mapping_version), 'NO_RULES_YET'),
    'manual override table has reviewed exceptions',
    'flag',
    CASE WHEN COUNT(*) = 0 THEN 1 ELSE 0 END,
    1,
    'An empty table is allowed and means no manual exception has been approved'
FROM `02-silver`.config_manual_geographic_match;

## Step 3 — Measure mapping coverage against Bronze

Coverage is a review signal, not an instruction to guess mappings. Each relevant Bronze domain is grouped once. The MGB table receives one grouped scan in this notebook, and only small grouped results are joined to configuration tables.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW silver_config_coverage_checks AS
WITH raw_dpwh_component_category AS (
    SELECT 'dpwh_projects' AS source_system, '' AS raw_category, source_infra_type, COUNT(*) AS source_rows
    FROM (
        SELECT ARRAY_DISTINCT(FILTER(TRANSFORM(
            SPLIT(COALESCE(GET_JSON_OBJECT(source_record_json, '$.componentCategories'), ''), ','),
            token -> TRIM(token)
        ), token -> token <> '')) AS source_infra_types
        FROM `01-bronze`.dpwh_projects
    ) AS source
    LATERAL VIEW EXPLODE(source.source_infra_types) exploded AS source_infra_type
    GROUP BY source_infra_type
),
raw_category AS (
    SELECT * FROM raw_dpwh_component_category
    UNION ALL
    SELECT 'flood_control_projects', COALESCE(TRIM(type_of_work), ''), COALESCE(TRIM(infra_type), ''), COUNT(*)
    FROM `01-bronze`.flood_control_projects
    GROUP BY COALESCE(TRIM(type_of_work), ''), COALESCE(TRIM(infra_type), '')
),
category_coverage AS (
    SELECT
        COUNT(*) AS raw_values,
        COUNT_IF(mapping.source_system IS NULL OR mapping.approval_status <> 'APPROVED') AS review_values,
        COALESCE(MAX(mapping.taxonomy_version), 'NO_RULES_YET') AS mapping_version
    FROM raw_category AS raw
    LEFT JOIN `02-silver`.config_project_category_mapping AS mapping
        ON raw.source_system = mapping.source_system
        AND raw.raw_category = mapping.raw_category
        AND raw.source_infra_type = mapping.source_infra_type
        AND mapping.is_active
),
raw_status AS (
    SELECT COALESCE(TRIM(status), '') AS source_status, COUNT(*) AS source_rows
    FROM `01-bronze`.dpwh_projects
    GROUP BY COALESCE(TRIM(status), '')
),
status_coverage AS (
    SELECT
        COUNT(*) AS raw_values,
        COUNT_IF(mapping.source_status IS NULL OR mapping.approval_status <> 'APPROVED') AS review_values,
        COALESCE(MAX(mapping.status_mapping_version), 'NO_RULES_YET') AS mapping_version
    FROM raw_status AS raw
    LEFT JOIN `02-silver`.config_project_status_mapping AS mapping
        ON mapping.source_system = 'dpwh_projects'
        AND raw.source_status = mapping.source_status
        AND mapping.is_active
),
raw_mgb AS (
    SELECT COALESCE(UPPER(TRIM(flood_susceptibility_code)), '') AS raw_code, COUNT(*) AS source_rows
    FROM `01-bronze`.flood_susceptibility
    GROUP BY COALESCE(UPPER(TRIM(flood_susceptibility_code)), '')
),
mgb_coverage AS (
    SELECT
        COUNT(*) AS raw_values,
        COUNT_IF(mapping.raw_susceptibility_code IS NULL OR mapping.approval_status <> 'APPROVED') AS review_values,
        COALESCE(MAX(mapping.mapping_version), 'NO_RULES_YET') AS mapping_version
    FROM raw_mgb AS raw
    LEFT JOIN `02-silver`.config_mgb_susceptibility_mapping AS mapping
        ON raw.raw_code = mapping.raw_susceptibility_code
        AND mapping.is_active
)
SELECT
    'completeness' AS data_quality_attribute,
    'config_project_category_mapping' AS table_name,
    mapping_version,
    'distinct Bronze source classifications have approved mappings' AS data_quality_check,
    'flag' AS action,
    review_values AS failed_rows,
    raw_values AS total_rows,
    'Unmapped and pending values remain visible for taxonomy review' AS details
FROM category_coverage
UNION ALL
SELECT
    'completeness',
    'config_project_status_mapping',
    mapping_version,
    'distinct Bronze project statuses have approved mappings',
    'flag',
    review_values,
    raw_values,
    'Unmapped and pending statuses remain visible for review'
FROM status_coverage
UNION ALL
SELECT
    'completeness',
    'config_mgb_susceptibility_mapping',
    mapping_version,
    'distinct Bronze MGB codes have approved mappings',
    'flag',
    review_values,
    raw_values,
    'Blank or new codes remain reviewable and are not guessed'
FROM mgb_coverage;

## Step 4 — Persist results and enforce the gate

One deterministic validation run ID represents the selected source snapshots, config versions, and complete result set. An exact rerun updates the same evidence rows. `PASS` and `FLAG` rows are saved for review. Any failed `stop` check raises an exception after persistence.

In [ ]:
CREATE OR REPLACE TEMPORARY VIEW silver_config_validation_results AS
WITH all_checks AS (
    SELECT * FROM silver_config_structural_checks
    UNION ALL
    SELECT * FROM silver_config_coverage_checks
),
latest_source_loads AS (
    SELECT table_name, snapshot_id
    FROM (
        SELECT
            table_name,
            snapshot_id,
            ROW_NUMBER() OVER (
                PARTITION BY table_name
                ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
            ) AS load_rank
        FROM `01-bronze`.load_log
        WHERE table_name IN ('dpwh_projects', 'flood_control_projects', 'flood_susceptibility')
          AND status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
    ) AS ranked_loads
    WHERE load_rank = 1
),
validation_run AS (
    SELECT
        SHA2(
            CONCAT_WS(
                '|',
                'silver_config_validation_v2',
                (
                    SELECT CONCAT_WS('||', SORT_ARRAY(COLLECT_LIST(CONCAT_WS(':', table_name, snapshot_id))))
                    FROM latest_source_loads
                ),
                CONCAT_WS(
                    '||',
                    SORT_ARRAY(COLLECT_LIST(CONCAT_WS(
                        ':',
                        table_name,
                        mapping_version,
                        data_quality_check,
                        CAST(failed_rows AS STRING),
                        CAST(total_rows AS STRING)
                    )))
                )
            ),
            256
        ) AS run_id,
        CURRENT_TIMESTAMP() AS run_ts
    FROM all_checks
)
SELECT
    validation_run.run_id,
    'silver_config' AS layer,
    checks.table_name,
    checks.mapping_version,
    checks.data_quality_attribute,
    checks.data_quality_check,
    CAST(checks.failed_rows AS BIGINT),
    CAST(checks.total_rows AS BIGINT),
    CASE
        WHEN checks.failed_rows = 0 THEN 'PASS'
        WHEN checks.action = 'stop' THEN 'FAIL'
        ELSE 'FLAG'
    END AS status,
    checks.action,
    checks.details,
    validation_run.run_ts
FROM all_checks AS checks
CROSS JOIN validation_run;

MERGE INTO `04-validation`.silver_config_dq_results AS target
USING silver_config_validation_results AS source
    ON target.run_id = source.run_id
    AND target.table_name = source.table_name
    AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [ ]:
WITH latest_run AS (
    SELECT run_id
    FROM `04-validation`.silver_config_dq_results
    WHERE layer = 'silver_config'
    GROUP BY run_id
    ORDER BY MAX(run_ts) DESC
    LIMIT 1
)
SELECT *
FROM `04-validation`.silver_config_dq_results
WHERE run_id = (SELECT run_id FROM latest_run)
ORDER BY action, table_name, data_quality_check;

In [ ]:
WITH latest_run AS (
    SELECT run_id
    FROM `04-validation`.silver_config_dq_results
    WHERE layer = 'silver_config'
    GROUP BY run_id
    ORDER BY MAX(run_ts) DESC
    LIMIT 1
),
blocking AS (
    SELECT *
    FROM `04-validation`.silver_config_dq_results
    WHERE run_id = (SELECT run_id FROM latest_run)
        AND status = 'FAIL'
        AND action = 'stop'
)
SELECT ASSERT_TRUE(
    COUNT(*) = 0,
    CONCAT(COUNT(*), ' Silver configuration checks blocked downstream use')
)
FROM blocking;

## Summary

The five configuration tables are safe for downstream consumption only when every `stop` check passes. The approved DPWH category taxonomy must select exactly one version and cannot fan out a source token. `FLAG` results remain expected for missing or future categories, status, place-alias, and manual exception decisions under review.